# Dead Neuron Recovery in Vision Transformers

Dead neurons in transformers live in the **FFN blocks**: the hidden layer after ReLU in `Linear(d_model, d_ff) → ReLU → Linear(d_ff, d_model)`. If a neuron in the `d_ff`-wide hidden layer has negative pre-activation for all inputs, it is dead.

We apply `DecomposedLinear` to the first FFN projection (the one before ReLU). The coupling matrix $P = I + \sum_i B_i B_i^\top$ couples dead FFN neurons to alive ones across all transformer blocks.

**Experiments** (all from the same killed-neuron checkpoint):
1. **Baseline** — standard ViT, no factors
2. **Baseline + Split** — add factors to FFN layers, merge+resplit every 10 epochs

In [ ]:
import sys
sys.path.insert(0, "../src")

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import copy

from WeightDecomp import DecomposedViT
from WeightDecomp.train_mnist import train_epoch, evaluate, reset_factor_optimizer_state

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
train_set = torchvision.datasets.MNIST(root="../data", train=True, transform=transform, download=False)
test_set = torchvision.datasets.MNIST(root="../data", train=False, transform=transform, download=False)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=256)

## Dead Neuron Detection & Killing

We track dead neurons in each FFN block's hidden layer. A neuron $j$ in `fc1` of block $b$ is dead if its pre-activation (before ReLU) is $< 0$ for all tokens across all samples.

In [ ]:
@torch.no_grad()
def count_dead_neurons(model, loader, device, max_batches=50):
    """Count dead neurons in each FFN block's fc1 (before ReLU).
    
    A neuron is dead if its pre-activation is < 0 for all tokens in all samples.
    Returns dict: block_idx -> (num_dead, total_neurons)
    """
    model.eval()
    ffn_layers = model.ffn_layers()
    
    # Hook to capture fc1 output (pre-ReLU) for each FFN block
    max_preact = [torch.full((l.out_features,), -float("inf"), device=device) for l in ffn_layers]
    activations = {}
    
    def make_hook(idx):
        def hook(module, input, output):
            # output shape: (batch, seq_len, d_ff)
            # Max over batch and seq_len dimensions
            channel_max = output.amax(dim=(0, 1))  # (d_ff,)
            max_preact[idx] = torch.maximum(max_preact[idx], channel_max)
        return hook
    
    handles = [layer.register_forward_hook(make_hook(i)) for i, layer in enumerate(ffn_layers)]
    
    for batch_idx, (images, _) in enumerate(loader):
        if batch_idx >= max_batches:
            break
        model(images.to(device))
    
    for h in handles:
        h.remove()
    
    return {i: (int((mp < 0).sum()), mp.numel()) for i, mp in enumerate(max_preact)}


def kill_neurons(model, frac=0.5, bias_val=-5.0, seed=123):
    """Kill a fraction of FFN hidden neurons by setting fc1 biases to large negative."""
    rng = torch.Generator().manual_seed(seed)
    for i, layer in enumerate(model.ffn_layers()):
        n = layer.out_features
        n_kill = int(n * frac)
        kill_idx = torch.randperm(n, generator=rng)[:n_kill]
        with torch.no_grad():
            layer.bias.data[kill_idx] = bias_val
        print(f"  Block {i}: killed {n_kill}/{n} FFN neurons (bias <- {bias_val})")

## Create Model, Kill Neurons, Warmup

In [ ]:
torch.manual_seed(42)

model = DecomposedViT(
    img_size=28, patch_size=7, in_channels=1, num_classes=10,
    d_model=64, n_heads=4, n_layers=4, d_ff=64,
).to(device)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"FFN layers: {len(model.ffn_layers())} blocks, d_ff=128\n")

print("Killing FFN neurons:")
kill_neurons(model, frac=0.5, bias_val=-5.0)

dead = count_dead_neurons(model, train_loader, device)
for i, (d, t) in dead.items():
    print(f"  Block {i}: {d}/{t} dead ({100*d/t:.0f}%)")

# Warmup
WARMUP_EPOCHS = 2
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

warmup_dead = [count_dead_neurons(model, train_loader, device)]
warmup_losses = []
warmup_accs = []

# print(f"\n--- Warmup ({WARMUP_EPOCHS} epochs) ---")
# for epoch in range(1, WARMUP_EPOCHS + 1):
#     loss, _ = train_epoch(model, train_loader, optimizer, criterion, device)
#     acc = evaluate(model, test_loader, device)
#     warmup_losses.append(loss)
#     warmup_accs.append(acc)
#     warmup_dead.append(count_dead_neurons(model, train_loader, device))
#     dead_str = ", ".join(f"B{i}: {d}/{t}" for i, (d, t) in warmup_dead[-1].items())
#     print(f"Epoch {epoch} | Loss: {loss:.4f} | Test: {100*acc:.2f}% | Dead: [{dead_str}]")

print("\nNow we fork.")

## Training Function

In [ ]:
def grad_norms_report(model):
    """Report per-layer gradient norms for W, A, C and shared B."""
    lines = []
    for i, layer in enumerate(model.ffn_layers()):
        w_grad = layer.W.grad
        if w_grad is None:
            lines.append(f"  L{i}: no grad")
            continue

        w_norm = w_grad.norm().item()
        a_norm = sum(A.grad.norm().item() for A in layer.As if A.grad is not None) if layer.As else 0
        c_norm = sum(C.grad.norm().item() for C in layer.Cs if C.grad is not None) if layer.Cs else 0

        lines.append(f"  L{i}: ||gW||={w_norm:.4f}  ||gA||={a_norm:.4f}  ||gC||={c_norm:.4f}")

    # Shared B norms
    if hasattr(model, 'shared_Bs') and model.shared_Bs:
        b_norms = [f"r{B.shape[0]}:{B.grad.norm().item():.4f}" for B in model.shared_Bs if B.grad is not None]
        if b_norms:
            lines.append(f"  shared_B: {', '.join(b_norms)}")
    return "\n".join(lines)


def continue_training(base_model, ranks=None, epochs=40, lr=1e-3, merge_resplit_every=None):
    model = copy.deepcopy(base_model)
    if ranks:
        model.split_all(ranks)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    dead_counts = [count_dead_neurons(model, train_loader, device)]
    train_losses = []
    test_accs = []

    for epoch in range(1, epochs + 1):
        if (merge_resplit_every and ranks
                and epoch > 1 and (epoch - 1) % merge_resplit_every == 0):
            model.merge_all(rerandomize_B=True)
            reset_factor_optimizer_state(optimizer, model)
            print(f"  [Merge + resplit at post-warmup epoch {epoch}]")

        loss, _ = train_epoch(model, train_loader, optimizer, criterion, device)
        acc = evaluate(model, test_loader, device)
        train_losses.append(loss)
        test_accs.append(acc)
        dead_counts.append(count_dead_neurons(model, train_loader, device))

        if epoch % 5 == 0 or epoch == 1:
            dc = dead_counts[-1]
            dc_str = ", ".join(f"B{i}: {d}/{t}" for i, (d, t) in dc.items())
            print(f"Epoch {epoch:3d} | Loss: {loss:.4f} | Test: {100*acc:.2f}% | Dead: [{dc_str}]")
            print(grad_norms_report(model))

    return {"dead_counts": dead_counts, "train_losses": train_losses, "test_accs": test_accs}

## Experiment 1: Baseline + Split

In [ ]:
POST_EPOCHS = 100
MERGE_EVERY = 10

RANKS = [4, 8, 16]
print(f"=== Baseline + Split (ranks={RANKS}, merge every {MERGE_EVERY}) ===")
split_result = continue_training(model, ranks=RANKS, epochs=POST_EPOCHS,
                                 merge_resplit_every=MERGE_EVERY)

## Experiment 2: Baseline

In [ ]:
print("=== Baseline ===")
baseline = continue_training(model, epochs=POST_EPOCHS)

## Results

In [ ]:
def full_dead_ts(warmup_dead, post_result, block_idx=0):
    warmup = [dc[block_idx][0] for dc in warmup_dead]
    post = [dc[block_idx][0] for dc in post_result["dead_counts"]]
    return warmup + post

def full_loss_ts(warmup_losses, post_result):
    return warmup_losses + post_result["train_losses"]

def full_acc_ts(warmup_accs, post_result):
    return warmup_accs + post_result["test_accs"]

experiments = {
    "Baseline": baseline,
    "Baseline + Split": split_result,
}

n_blocks = len(model.ffn_layers())
split_epoch = WARMUP_EPOCHS
merge_epochs = [WARMUP_EPOCHS + e for e in range(MERGE_EVERY, POST_EPOCHS, MERGE_EVERY)]

def add_event_lines(ax):
    ax.axvline(x=split_epoch, color="gray", linestyle="--", alpha=0.7, label="Split")
    for i, me in enumerate(merge_epochs):
        ax.axvline(x=me, color="red", linestyle=":", alpha=0.4,
                   label="Merge+resplit" if i == 0 else None)

# Layout: top row = dead neurons per block, bottom row = loss + accuracy + total dead
fig, axes = plt.subplots(2, max(n_blocks, 3), figsize=(5 * max(n_blocks, 3), 8))

# Top row: dead neurons per FFN block
for b in range(n_blocks):
    ax = axes[0, b]
    for name, result in experiments.items():
        ts = full_dead_ts(warmup_dead, result, b)
        ax.plot(range(len(ts)), ts, label=name, marker="o", markersize=2)
    add_event_lines(ax)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Dead Neurons")
    ax.set_title(f"FFN Block {b} (d_ff=128)")
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)

# Bottom left: train loss
ax = axes[1, 0]
for name, result in experiments.items():
    ts = full_loss_ts(warmup_losses, result)
    ax.plot(range(1, len(ts) + 1), ts, label=name, marker="o", markersize=2)
add_event_lines(ax)
ax.set_xlabel("Epoch")
ax.set_ylabel("Train Loss")
ax.set_title("Training Loss")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Bottom middle: test accuracy
ax = axes[1, 1]
for name, result in experiments.items():
    ts = full_acc_ts(warmup_accs, result)
    ax.plot(range(1, len(ts) + 1), [100*a for a in ts], label=name, marker="o", markersize=2)
add_event_lines(ax)
ax.set_xlabel("Epoch")
ax.set_ylabel("Test Accuracy (%)")
ax.set_title("Test Accuracy")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Bottom right: total dead across all blocks
ax = axes[1, 2]
for name, result in experiments.items():
    total = [sum(full_dead_ts(warmup_dead, result, b)[ep]
                 for b in range(n_blocks))
             for ep in range(len(full_dead_ts(warmup_dead, result, 0)))]
    ax.plot(range(len(total)), total, label=name, marker="o", markersize=2)
add_event_lines(ax)
ax.set_xlabel("Epoch")
ax.set_ylabel("Total Dead Neurons")
ax.set_title("Total Dead (all FFN blocks)")
ax.legend(fontsize=7)
ax.grid(True, alpha=0.3)

# Hide extra axes
for col in range(3, axes.shape[1]):
    axes[1, col].set_visible(False)

plt.tight_layout()
plt.savefig("vit_dead_neuron_recovery.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary

Both models are functionally identical at the split point. The decomposition adds factors with $C = 0$ to the FFN's first projection in each transformer block.

- **Baseline**: FFN neurons that are dead stay dead — their gradient rows in $G$ are zero, and standard gradient descent cannot revive them.
- **Baseline + Split**: the coupling channel $P = I + \sum_i B_i B_i^\top$ on the FFN projection mixes gradients from alive neurons into dead neuron directions. Periodic merge+resplit refreshes the coupling.

This confirms the mechanism is architecture-agnostic: it works wherever there is a `Linear → ReLU` pattern, including transformer FFN blocks.